# 🕵️ Dia 12 — Triagem de logs de autenticação e a linha do tempo de uma intrusão SSH

**Investigador Cyber 90 Dias** · Fase 1 · Rank Jutsu `D`

> Baseado no *Day 12* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-1-foundations/day-12.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Terça de manhã. No `auth.log` do servidor do portal da Serra Azul, uma linha fora do padrão: *domingo, 22h47, `Accepted password for juliano from 198.51.100.200`*, com duas falhas antes e dois `sudo` depois. O Juliano só entra com chave, pela VPN.

Rafa: *"Escreve também o que derrubaria isso. Um fato só."*

Neste notebook você vai:

1. **Ler os dois formatos de carimbo** (syslog clássico e RFC 3339) e normalizar tudo em UTC.
2. **Transformar o fluxograma de triagem** em uma função que classifica cada linha.
3. **Gerar a linha do tempo** em CSV e em texto, para o caso do original e para a Serra Azul.
4. **Jogar "ruído ou achado?"** e escrever a avaliação com a hipótese concorrente.

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Tudo funciona **sem internet**; quando uma célula tenta a rede, ela tem plano B embutido.

## 1. Dois formatos de carimbo, um problema

`Mar 10 02:14:09` não tem **ano** nem **fuso**. `2026-10-04T22:47:25.330871-03:00` tem os dois. A função abaixo lê os dois, mas exige que você **diga** o ano e o fuso do formato clássico: é exatamente o que o Checkpoint pede para você saber explicar.

In [ ]:
import re
from datetime import datetime, timezone, timedelta

def ler_carimbo(linha, ano_contexto=None, fuso_contexto=None):
    """
    Devolve (datetime em UTC, resto da linha).
    - RFC 3339: tudo está na própria linha.
    - syslog clássico: ano e fuso PRECISAM vir de fora (ano_contexto, fuso_contexto).
    """
    primeiro = linha.split(" ", 1)[0]
    if re.match(r"\d{4}-\d{2}-\d{2}T", primeiro):            # começa com 2026-10-04T...
        dt = datetime.fromisoformat(primeiro)                 # Python entende o -03:00 sozinho
        return dt.astimezone(timezone.utc), linha.split(" ", 1)[1]
    # Formato clássico: "Mar 10 02:14:09 host ..."
    if ano_contexto is None or fuso_contexto is None:
        raise ValueError("Linha syslog clássica: informe o ano e o fuso de onde você os tirou.")
    texto = " ".join(linha.split()[:3])                       # "Mar 10 02:14:09"
    dt = datetime.strptime(f"{ano_contexto} {texto}", "%Y %b %d %H:%M:%S")
    dt = dt.replace(tzinfo=fuso_contexto)                      # cola o fuso que VOCÊ confirmou
    return dt.astimezone(timezone.utc), linha.split(" ", 3)[3]

# Teste: a mesma função nos dois formatos
print(ler_carimbo("2026-10-04T22:47:25.330871-03:00 sa-web01 sshd[3120]: Accepted ...")[0])
print(ler_carimbo("Mar 10 02:31:55 lab-web01 sshd[1902]: Accepted ...",
                  ano_contexto=2026, fuso_contexto=timezone.utc)[0])
try:
    ler_carimbo("Mar 10 02:31:55 lab-web01 sshd[1902]: Accepted ...")
except ValueError as e:
    print("Sem contexto ->", e)

## 2. O fluxograma de triagem, como função

Uma regex por tipo de evento e a mesma ordem de perguntas do diagrama do README.

In [ ]:
# Padrões das linhas do sshd/sudo. (?:...)? = pedaço opcional.
RX = {
    "fail":    re.compile(r"Failed password for (?:invalid user )?(?P<user>\S+) from (?P<ip>\S+)"),
    "ok":      re.compile(r"Accepted (?P<metodo>password|publickey) for (?P<user>\S+) from (?P<ip>\S+)"),
    "sudo":    re.compile(r"sudo:\s+(?P<user>\S+) : .*COMMAND=(?P<cmd>.*)"),
    "aberta":  re.compile(r"session opened for user (?P<user>[^(\s]+)"),
    "fechada": re.compile(r"session closed for user (?P<user>\S+)"),
}

def eventos(texto, **contexto):
    """Lê o log e devolve uma lista de eventos normalizados (hora UTC, tipo, user, ip, extra)."""
    saida = []
    for n, linha in enumerate(texto.strip().splitlines(), start=1):
        if "Invalid user" in linha and "Failed" not in linha:
            continue                                        # a linha "Failed ... invalid user" já conta
        utc, resto = ler_carimbo(linha, **contexto)
        for tipo, rx in RX.items():
            m = rx.search(resto)
            if m:
                d = m.groupdict()
                saida.append({"n": n, "utc": utc, "tipo": tipo, "user": d.get("user"),
                              "ip": d.get("ip", "-"), "extra": d.get("metodo") or d.get("cmd") or ""})
                break
    return saida

def triar(evs, base):
    """
    base = {conta: {"ips": {...}, "metodo": "publickey", "horas": range(8, 19)}} (horas na hora LOCAL)
    Aplica o fluxograma e devolve uma lista de (evento, veredito).
    """
    falhas_por_ip = {}
    resultado = []
    for e in evs:
        if e["tipo"] == "fail":
            falhas_por_ip[e["ip"]] = falhas_por_ip.get(e["ip"], 0) + 1
            resultado.append((e, "ruído"))
        elif e["tipo"] == "ok":
            b = base.get(e["user"], {})
            local = e["utc"] + b.get("utc_offset", timedelta(0))   # volta para a hora local da conta
            desvios = []
            if e["ip"] not in b.get("ips", set()):
                desvios.append("origem")
            if b.get("metodo") and e["extra"] != b["metodo"]:
                desvios.append("método")
            if local.hour not in b.get("horas", range(24)):
                desvios.append("horário")
            if not desvios:
                resultado.append((e, "linha de base"))
            elif falhas_por_ip.get(e["ip"]):
                resultado.append((e, f"ACHADO PRIORITÁRIO ({', '.join(desvios)}; {falhas_por_ip[e['ip']]} falha(s) antes)"))
            else:
                resultado.append((e, f"achado: login incomum ({', '.join(desvios)})"))
        else:
            resultado.append((e, "contexto"))               # sudo e sessões: o que veio depois
    return resultado

print("Funções prontas.")

### 2a. O caso do original (`lab-web01`, servidor em UTC)

In [ ]:
AUTH_LAB = 'Mar 10 02:14:07 lab-web01 sshd[1811]: Invalid user admin from 203.0.113.45 port 50122\nMar 10 02:14:09 lab-web01 sshd[1811]: Failed password for invalid user admin from 203.0.113.45 port 50122 ssh2\nMar 10 02:14:12 lab-web01 sshd[1815]: Invalid user oracle from 203.0.113.45 port 50188\nMar 10 02:14:14 lab-web01 sshd[1815]: Failed password for invalid user oracle from 203.0.113.45 port 50188 ssh2\nMar 10 02:14:18 lab-web01 sshd[1819]: Failed password for root from 203.0.113.45 port 50240 ssh2\nMar 10 02:14:22 lab-web01 sshd[1823]: Failed password for root from 203.0.113.45 port 50301 ssh2\nMar 10 02:31:40 lab-web01 sshd[1902]: Failed password for deploy from 198.51.100.9 port 41822 ssh2\nMar 10 02:31:47 lab-web01 sshd[1902]: Failed password for deploy from 198.51.100.9 port 41822 ssh2\nMar 10 02:31:55 lab-web01 sshd[1902]: Accepted password for deploy from 198.51.100.9 port 41822 ssh2\nMar 10 02:31:55 lab-web01 sshd[1902]: pam_unix(sshd:session): session opened for user deploy(uid=1001) by (uid=0)\nMar 10 02:33:02 lab-web01 sudo:   deploy : TTY=pts/0 ; PWD=/home/deploy ; USER=root ; COMMAND=/usr/bin/cat /etc/shadow\nMar 10 02:35:40 lab-web01 sshd[1902]: pam_unix(sshd:session): session closed for user deploy\nMar 10 09:02:13 lab-web01 sshd[2410]: Accepted publickey for analyst from 192.0.2.10 port 55012 ssh2: ED25519 SHA256:q3Vd1xExampleFingerprintOnlyForLab0000000\nMar 10 09:02:13 lab-web01 sshd[2410]: pam_unix(sshd:session): session opened for user analyst(uid=1000) by (uid=0)\n'

# Contexto fornecido no Passo 1 do README
BASE_LAB = {
    "deploy":  {"ips": {"192.0.2.50"}, "metodo": "publickey"},
    "analyst": {"ips": {"192.0.2.10"}, "metodo": "publickey", "horas": range(8, 19)},
}
evs_lab = eventos(AUTH_LAB, ano_contexto=2026, fuso_contexto=timezone.utc)   # ano e fuso: do enunciado
for e, v in triar(evs_lab, BASE_LAB):
    print(f"{e['utc']:%Y-%m-%dT%H:%M:%SZ}  {e['tipo']:<8} {e['user'] or '':<8} {e['ip']:<14} -> {v}")

### 2b. ➕ O `auth.log` do portal da Serra Azul

[`caso-continuo/dados/dia-12/auth-sa-web01.log`](https://github.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/blob/main/caso-continuo/dados/dia-12/auth-sa-web01.log). RFC 3339 com `-03:00`: não precisa de contexto de ano e fuso.

In [ ]:
# ------------------------------------------------------------
# Carrega o arquivo do caso: primeiro tenta o repositório no GitHub,
# depois cai na cópia embutida (offline-first: o notebook nunca trava).
# ------------------------------------------------------------
import urllib.request                      # cliente HTTP da biblioteca padrão

URL = "https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/dia-12/auth-sa-web01.log"
EMBUTIDO = '2026-10-03T03:12:07.118204-03:00 sa-web01 sshd[1811]: Invalid user admin from 203.0.113.9 port 50122\n2026-10-03T03:12:09.401121-03:00 sa-web01 sshd[1811]: Failed password for invalid user admin from 203.0.113.9 port 50122 ssh2\n2026-10-03T03:12:14.552310-03:00 sa-web01 sshd[1815]: Invalid user ubuntu from 203.0.113.9 port 50188\n2026-10-03T03:12:16.020934-03:00 sa-web01 sshd[1815]: Failed password for invalid user ubuntu from 203.0.113.9 port 50188 ssh2\n2026-10-03T03:12:20.700012-03:00 sa-web01 sshd[1819]: Failed password for root from 203.0.113.9 port 50240 ssh2\n2026-10-03T22:51:07.300455-03:00 sa-web01 sshd[2077]: Invalid user serraazul from 203.0.113.77 port 40112\n2026-10-03T22:51:09.881203-03:00 sa-web01 sshd[2077]: Failed password for invalid user serraazul from 203.0.113.77 port 40112 ssh2\n2026-10-04T22:47:10.114092-03:00 sa-web01 sshd[3120]: Failed password for juliano from 198.51.100.200 port 51522 ssh2\n2026-10-04T22:47:16.902331-03:00 sa-web01 sshd[3120]: Failed password for juliano from 198.51.100.200 port 51522 ssh2\n2026-10-04T22:47:25.330871-03:00 sa-web01 sshd[3120]: Accepted password for juliano from 198.51.100.200 port 51522 ssh2\n2026-10-04T22:47:25.401116-03:00 sa-web01 sshd[3120]: pam_unix(sshd:session): session opened for user juliano(uid=1002) by (uid=0)\n2026-10-04T22:49:02.775510-03:00 sa-web01 sudo:  juliano : TTY=pts/0 ; PWD=/home/juliano ; USER=root ; COMMAND=/usr/sbin/nginx -t\n2026-10-04T22:49:05.120998-03:00 sa-web01 sudo:  juliano : TTY=pts/0 ; PWD=/home/juliano ; USER=root ; COMMAND=/usr/bin/systemctl reload nginx\n2026-10-04T22:58:40.003214-03:00 sa-web01 sshd[3120]: pam_unix(sshd:session): session closed for user juliano\n2026-10-05T08:02:13.550021-03:00 sa-web01 sshd[3388]: Accepted publickey for juliano from 198.51.100.130 port 55012 ssh2: ED25519 SHA256:sAzUlExampleFingerprintOnlyForLab00000000\n2026-10-05T08:02:13.611904-03:00 sa-web01 sshd[3388]: pam_unix(sshd:session): session opened for user juliano(uid=1002) by (uid=0)\n2026-10-05T09:31:44.901223-03:00 sa-web01 sshd[3502]: Failed password for root from 203.0.113.9 port 61002 ssh2\n2026-10-05T09:31:49.004512-03:00 sa-web01 sshd[3502]: Failed password for root from 203.0.113.9 port 61002 ssh2\n'

try:
    with urllib.request.urlopen(URL, timeout=8) as resp:   # timeout curto: sem rede, desiste logo
        AUTH_SA = resp.read().decode("utf-8")
    origem = "GitHub"
except Exception as erro:                  # sem internet, repo privado, URL mudou...
    AUTH_SA = EMBUTIDO                   # ...plano B: a cópia que veio dentro do notebook
    origem = f"cópia embutida ({type(erro).__name__})"

print(f"Carregado de: {origem} — {len(AUTH_SA.splitlines())} linhas")

In [ ]:
import hashlib
print("SHA-256:", hashlib.sha256(AUTH_SA.encode()).hexdigest())

BRT = timedelta(hours=-3)                                     # para voltar de UTC à hora local
BASE_SA = {"juliano": {"ips": {"198.51.100.130"}, "metodo": "publickey",
                       "horas": range(8, 19), "utc_offset": BRT}}
IPS_DO_CASO = {"203.0.113.77": "IP de serraazul-pix.example (Dias 5 e 11)"}

evs_sa = eventos(AUTH_SA)
for e, v in triar(evs_sa, BASE_SA):
    if e["ip"] in IPS_DO_CASO:                                # ruído que não é ruído
        v = f"ACHADO LIGADO AO CASO: {IPS_DO_CASO[e['ip']]}, tentou '{e['user']}'"
    print(f"{e['utc']:%Y-%m-%dT%H:%M:%SZ}  {e['tipo']:<8} {e['user'] or '':<10} {e['ip']:<15} {e['extra'][:28]:<28} -> {v}")

> 🧠 Repare no que veio **depois** do login de domingo: `nginx -t` e `systemctl reload nginx`. É manutenção comum. Não houve leitura de `/etc/shadow`, nem usuário novo, nem chave nova. Isso não inocenta nem condena: é o que o log mostra, e precisa entrar na avaliação.

## 3. A linha do tempo: CSV em UTC e um desenho em texto

In [ ]:
import csv, io
from pathlib import Path

def para_csv(evs, host):
    """Gera o CSV do Passo 5 (com session_closed), hora em ISO 8601 UTC."""
    buf = io.StringIO()
    w = csv.writer(buf)
    w.writerow(["time", "host", "event", "user", "src_ip", "linha_origem"])
    nomes = {"fail": "ssh_fail", "fechada": "session_closed"}
    for e in evs:
        if e["tipo"] == "aberta":
            continue                                           # o original não lista a abertura (ela = o Accepted)
        if e["tipo"] == "ok":
            ev = "ssh_success_" + e["extra"]
        elif e["tipo"] == "sudo":
            ev = "sudo:" + e["extra"]
        else:
            ev = nomes[e["tipo"]]
        w.writerow([f"{e['utc']:%Y-%m-%dT%H:%M:%SZ}", host, ev, e["user"], e["ip"], e["n"]])
    return buf.getvalue()

Path("notas").mkdir(exist_ok=True)
CSV_SA = para_csv(evs_sa, "sa-web01")
Path("notas/isca-dia12-timeline.csv").write_text(CSV_SA, encoding="utf-8")
Path("notas/day12-timeline.csv").write_text(para_csv(evs_lab, "lab-web01"), encoding="utf-8")
print(CSV_SA)
print("Linhas de evento:", len(CSV_SA.strip().splitlines()) - 1)

In [ ]:
def desenho(evs, ips_caso):
    """Linha do tempo em texto: um símbolo por tipo, agrupada por dia (UTC)."""
    simbolo = {"fail": "·", "ok": "●", "sudo": "▲", "aberta": "[", "fechada": "]"}
    dia_atual = None
    for e in evs:
        dia = f"{e['utc']:%Y-%m-%d}"
        if dia != dia_atual:
            print(f"\n── {dia} (UTC) " + "─" * 40)
            dia_atual = dia
        marca = simbolo[e["tipo"]]
        if e["ip"] in ips_caso:
            marca = "◆"                                          # roxo no desenho do README
        print(f"  {e['utc']:%H:%M:%S}  {marca} {e['tipo']:<8} {e['user'] or ''} {e['ip'] if e['ip'] != '-' else ''} {e['extra'][:30]}")
    print("\nLegenda: · falha  ● sucesso  ▲ sudo  [ ] sessão  ◆ IP do caso")

desenho(evs_sa, IPS_DO_CASO)

## 4. 🎮 Ruído ou achado?

Dez linhas. Para cada uma, responda **R** (ruído / linha de base) ou **A** (achado que entra no relatório). `MODO_JOGO = False` mostra só o gabarito.

In [ ]:
MODO_JOGO = True

JOGO = [
    ("Failed password for invalid user admin from 203.0.113.9", "R", "Lista genérica de usernames, nunca entra."),
    ("Failed password for invalid user serraazul from 203.0.113.77", "A", "IP do site falso, username com o nome da marca, 10 min depois do espelhamento."),
    ("Accepted publickey for juliano from 198.51.100.130 às 08:02 BRT", "R", "Origem, método e horário da linha de base."),
    ("Accepted password for juliano from 198.51.100.200 às 22:47 BRT, 2 falhas antes", "A", "Três desvios + falhas antes. Achado, ainda que a avaliação seja cautelosa."),
    ("sudo: juliano ... COMMAND=/usr/bin/systemctl reload nginx (dentro da sessão de domingo)", "A", "Sozinho seria manutenção; dentro da sessão suspeita, é o 'o que veio depois' e entra na linha do tempo."),
    ("Failed password for root from 203.0.113.9 (segunda, 09:31)", "R", "Mesmo scanner do sábado. Ruído."),
    ("Accepted password for deploy from 198.51.100.9 (lab-web01)", "A", "Conta de serviço que só usa chave, de outra origem."),
    ("sudo: deploy ... COMMAND=/usr/bin/cat /etc/shadow", "A", "Leitura de todos os hashes de senha."),
    ("Accepted publickey for analyst from 192.0.2.10 às 09:02 (lab-web01)", "R", "Bate com a linha de base do administrador."),
    ("Failed password for root from 203.0.113.45 port 50301 (lab-web01)", "R", "Varredura genérica sem sucesso."),
]
acertos = 0
for i, (linha, certo, porque) in enumerate(JOGO, start=1):
    print(f"\n[{i:02d}/{len(JOGO)}] {linha}")
    if MODO_JOGO:
        r = ""
        while r not in ("R", "A"):
            r = input("   Ruído (R) ou Achado (A)? ").strip().upper()[:1]
        acertos += (r == certo)
        print("   ✅" if r == certo else f"   ❌ Era {certo}.")
    print("   Por quê:", porque)
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(JOGO)}", "— Checkpoint BR cumprido." if acertos >= 8 else "— Meta: 8.")

## 5. 📝 A avaliação, com o fato que a derruba

Rafa não aceita avaliação sem hipótese concorrente. O formulário abaixo já vem com um texto de exemplo; **reescreva com as suas palavras**.

In [ ]:
#@title Avaliação do login de domingo { display-mode: "form" }
avaliacao = "Um login na conta juliano em 2026-10-05T01:47:25Z (22:47 BRT de 4/out) desviou da linha de base em origem (198.51.100.200), método (senha) e horário, após 2 falhas da mesma origem."  #@param {type:"string"}
o_que_veio_depois = "Dois sudo de manutenção (nginx -t, systemctl reload nginx); sessão fechada às 01:58:40Z. Nenhuma leitura de /etc/shadow, usuário ou chave nova no log."  #@param {type:"string"}
h1 = "O próprio Juliano, de casa, fora do padrão."  #@param {type:"string"}
h2 = "Alguém com a senha dele testando o acesso sem chamar atenção."  #@param {type:"string"}
confianca = "baixa a moderada para uso não autorizado"  #@param {type:"string"}
fato_que_derruba = "Juliano confirma que entrou de casa no domingo e há registro de mudança/plantão do reload do nginx nesse horário."  #@param {type:"string"}
achado_ligado = "203.0.113.77 (site falso) tentou o username 'serraazul' em 2026-10-04T01:51:09Z, 10 min após espelhar o portal; sem sucesso."  #@param {type:"string"}

texto = f"""# Avaliação — auth.log de sa-web01 (Dia 12)

## Login de domingo
{avaliacao}

**O que veio depois:** {o_que_veio_depois}

- H1: {h1}
- H2: {h2}

Avaliação: possível uso não autorizado da conta juliano. Confiança: {confianca}; precisa de confirmação de que {fato_que_derruba[0].lower() + fato_que_derruba[1:]}

**Fato que derrubaria H2:** {fato_que_derruba}

## Achado ligado ao caso
{achado_ligado}

## Ruído
203.0.113.9: varredura genérica (admin, ubuntu, root), nenhum sucesso.
"""
Path("notas/isca-dia12-avaliacao.md").write_text(texto, encoding="utf-8")
print(texto)
try:
    from google.colab import files
    files.download("notas/isca-dia12-avaliacao.md")
    files.download("notas/isca-dia12-timeline.csv")
except ImportError:
    print("(Fora do Colab: os arquivos ficaram em notas/)")

### (Opcional) Na sua máquina de laboratório

In [ ]:
!journalctl -u ssh --since "today" -o short-iso-precise | tail -5
!last -F | head -5
print("(Linhas com '!' só rodam num Linux com systemd.)")

## ✅ Antes de fechar o Dia 12

- [ ] Jogo com pelo menos 8/10
- [ ] `notas/isca-dia12-timeline.csv` com 13 eventos em UTC
- [ ] `notas/isca-dia12-avaliacao.md` com H1, H2 e o fato que derruba
- [ ] `day12-timeline.csv` e `day12-timeline.png` do original (README, Passos 1 a 6)
- [ ] `status` do dia 12 = `estudado` no `curriculo/mapa-dos-dias.csv`

**Amanhã (Dia 13):** automatizar uma etapa de reconhecimento em Bash. Os domínios imitação mudam de um dia para o outro; você vai provar isso com dois retratos de DNS.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>